In [1]:
import serial
import serial.tools.list_ports
import time

In [12]:
class ArduinoSerial:
    """Class for Arduino communication over USB or Bluetooth (serial connection)."""

    def __init__(self, port=None, baudrate=9600, timeout=1, 
                 bytesize=serial.EIGHTBITS, parity=serial.PARITY_NONE,
                 stopbits=serial.STOPBITS_ONE, xonxoff=False, rtscts=False,
                 inter_char_timeout=None):
        """
        Initialize the serial connection settings.

        :param port: Serial port (e.g., "/dev/ttyUSB0", "/dev/ttyS0", "COM3"). If None, auto-detects.
        :param baudrate: Baud rate (default 9600)
        :param timeout: Timeout for reading data (default 1s)
        :param bytesize: Data bit size (default 8 bits)
        :param parity: Parity check (default None)
        :param stopbits: Stop bits (default 1)
        :param xonxoff: Software flow control (default False)
        :param rtscts: Hardware flow control (default False)
        :param inter_char_timeout: Timeout between characters (default None)
        """
        self.port = port if port else self.auto_detect_port()
        self.baudrate = baudrate
        self.timeout = timeout
        self.bytesize = bytesize
        self.parity = parity
        self.stopbits = stopbits
        self.xonxoff = xonxoff
        self.rtscts = rtscts
        self.inter_char_timeout = inter_char_timeout
        self.connection = None

    @staticmethod
    def auto_detect_port():
        """Automatically detect an available Arduino port."""
        ports = list(serial.tools.list_ports.comports())
        for port in ports:
            if ("Arduino" in port.description 
                or "tty" in port.device 
                or "COM" in port.device
                or "cu.usb" in port.device):
                print(f"Auto-detected Arduino on {port.device}")
                return port.device
        raise Exception("No Arduino detected. Please specify the port manually.")

    def connect(self):
        """Establish connection with the Arduino."""
        if not self.port:
            raise Exception("No available serial port found.")
        
        self.connection = serial.Serial(
            self.port, self.baudrate, timeout=self.timeout,
            bytesize=self.bytesize, parity=self.parity,
            stopbits=self.stopbits, xonxoff=self.xonxoff,
            rtscts=self.rtscts, interCharTimeout=self.inter_char_timeout
        )
        time.sleep(2)  # Allow Arduino to initialize
        print(f"Connected to Arduino on {self.port}")

    def read_data(self):
        """Read and return data from the serial port."""
        if self.connection and self.connection.in_waiting:
            return self.connection.readline().decode().strip()
        return None

    def write_data(self, data):
        """Send data to the Arduino."""
        self.connection.reset_input_buffer()
        if self.connection:
            self.connection.write(data.encode())

    def disconnect(self):
        """Close the connection."""
        if self.connection:
            self.connection.close()
            print("Disconnected from Arduino")

In [15]:
arduino = ArduinoSerial()

Auto-detected Arduino on /dev/cu.usbmodem1201


In [17]:
try:
    loop_duration = 5
    start_time = time.time()
    arduino.connect()
    arduino.write_data("s")
    while time.time() - start_time < loop_duration:
        r = arduino.read_data()
        if r is not None:
            print(r)
    arduino.write_data("p")
except KeyboardInterrupt:
    print("Exiting...")
    arduino.write_data("p")
finally:
    arduino.disconnect()

Connected to Arduino on /dev/cu.usbmodem1201
1,20.61
3,22.04
5,20.76
9,20.66
11,19.47
13,19.50
15,20.66
17,20.66
19,19.88
25,19.61
35,19.14
46,19.06
57,19.14
67,18.69
77,18.28
87,18.28
98,18.68
108,17.82
120,17.82
131,17.82
143,17.83
154,17.83
165,17.82
178,18.23
189,18.23
200,17.76
211,17.76
223,17.37
234,17.76
246,17.76
257,17.76
269,17.32
280,17.73
291,17.32
302,17.71
315,17.71
326,18.13
337,17.66
349,17.68
360,17.68
371,17.68
383,17.68
395,17.66
406,17.68
417,18.07
428,18.07
440,17.61
452,17.63
463,17.61
474,17.61
486,18.02
497,18.02
508,18.02
521,18.02
532,18.02
543,18.02
554,18.02
566,18.44
578,18.33
589,18.85
600,18.85
612,19.25
623,20.54
635,20.54
652,106.17
664,106.27
676,106.27
689,105.45
701,104.59
714,105.76
726,105.88
739,105.88
751,105.88
763,105.88
776,105.86
789,104.59
801,105.88
813,105.86
826,106.27
838,105.88
851,105.36
864,105.86
876,105.86
888,105.45
900,105.45
913,104.19
926,104.19
938,105.45
951,105.45
963,105.46
975,105.45
989,105.45
1001,104.17
1009,20.93
1021,